# Lab 52087: Inversion on Neural Fields

In this lab, we will explore the concept of inversion and differentiability in neural fields, using images as an example.
In this part, we will explore how to invert a trained neural field (like a PE-MLP or SIREN) that maps from spatial coordinates (x, y) to RGB color values. Specifically, for a given target color, we aim to find the input coordinate that the neural field would map as closely as possible to a tartget (color, patch or something else).

## Task 1:
Inversion on Neural Fields with a Trained (Frozen) Model (use your trained PE-MLP or SIREN from last LAB)

The goal of this task is to perform inversion on a neural field that has already been trained (and is now frozen). 
The idea is to find an input coordinate (x, y) that, when passed through the neural field, produces a given target color.

Step-by-step procedure:

1. Train your neural field model (e.g., SIREN) on the `map-saturation.png` image. 
   After training, visualize the reconstruction to check that the model has learned the mapping correctly.

2. Select a random coordinate (pixel) from the image and use its color as the target color. 
   This will be the reference value we are trying to recover via inversion.

3. Randomly initialize a coordinate (x, y) in the normalized range [-1, 1]. 
   This coordinate is learnable, while the neural field model’s parameters remain fixed.

4. Define an optimization loop to adjust only the (x, y) coordinate:
   - For each iteration, compute the model’s predicted color from the current coordinate.
   - Calculate the loss between the predicted color and the target color (e.g., using MSE).
   - Backpropagate and update only the coordinate to minimize the loss.
   - Ensure the neural field model remains frozen (no parameter updates).

5. Visualize the inversion process:
   - Track and plot the loss over time.
   - Visualize the trajectory of the coordinate in pixel space to see how it moves towards the solution.

6. Experiment with different random initializations of the coordinate, and observe if and how they lead to different local solutions.
   - Discuss any differences or interesting behaviors you find when starting from different points.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from SIREN import SIREN, load_image, prepare_dataset, train_model

In [2]:
# 1. Train
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

img_path = "map-saturation.png"
resolution = 512 
img_np = load_image(img_path, resize=(resolution, resolution))

coords, colors = prepare_dataset(img_np, device)

hidden_dim_siren = 256
num_layers_siren = 5
omega_0 = 30.0
# Train and plot the training dynamics of the SIREN neural field model.
model = SIREN(in_dim=2, hidden_dim=hidden_dim_siren, out_dim=3, num_layers=num_layers_siren, omega_0=omega_0).to(device)
num_iters = 2000
lr = 1e-4
trained_model, history = train_model(model, coords, colors, num_iters=num_iters, lr=lr, loss_type="mse", log_every=200)


# Show the final result of the SIREN neural field model.
trained_model.eval()
with torch.no_grad():
    pred_colors = trained_model(coords).cpu().numpy()
recon_img = pred_colors.reshape(resolution, resolution, 3)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)            
plt.title("Original Image")
plt.imshow(img_np)
plt.axis('off')
plt.subplot(1, 2, 2)    
plt.title("SIREN Reconstructed Image")
plt.imshow(recon_img)
plt.axis('off')
plt.show()

Using device: cuda
Iter 1/2000, Loss: 0.078368, PSNR: 11.06 dB


KeyboardInterrupt: 

In [ ]:
# 2. Select a random coordinate (pixel) from the image and use its color as the target color. This will be the reference value we are trying to recover via inversion.
rand_idx = np.random.randint(0, coords.shape[0])
target_coord = coords[rand_idx:rand_idx+1, :].to(device)  # Shape: (1, 2)
target_color = colors[rand_idx:rand_idx+1, :].to(device)  # Shape: (1, 3)

In [ ]:
num_seeds = 10
learned_coords = torch.empty((num_seeds, 2), device=device).uniform_(-1, 1).requires_grad_(True)
print(f"Target coordinate (pixel): {target_coord.cpu().numpy()}")
print(f"Initial learned coordinates:\n{learned_coords.detach().cpu().numpy()}")
optimizer = optim.Adam([learned_coords], lr=1e-2)

history_coords = []
num_inversion_iters = 2000

for iter in range(num_inversion_iters):
    optimizer.zero_grad()
    
    # Le modèle prédit les couleurs pour les N tentatives
    pred_colors = trained_model(learned_coords) 
    
    # Loss calculée par rapport à la cible unique (broadcast)
    loss = F.mse_loss(pred_colors, target_color.expand(num_seeds, -1))
    
    loss.backward()
    optimizer.step()
    
    with torch.no_grad():
        learned_coords.clamp_(-1, 1)
        history_coords.append(learned_coords.clone().cpu().numpy())

    if (iter + 1) % 200 == 0:
        print(f"Iteration {iter+1}, Loss moyenne: {loss.item():.6f}")

# Visualisation (Tâche 5 & 6)
history_coords = np.array(history_coords) # Shape: (iters, num_seeds, 2)

plt.imshow(recon_img, extent=[-1, 1, 1, -1]) # Utilisation de l'étendue pour matcher les coords
for i in range(num_seeds):
    plt.plot(history_coords[:, i, 0], history_coords[:, i, 1], label=f'Seed {i}')
plt.scatter(target_coord.cpu()[0,0], target_coord.cpu()[0,1], c='red', marker='x', s=100, label='Cible réelle')
plt.legend()
plt.title("Trajectoires d'inversion dans l'espace des coordonnées")
plt.show()

## Task 2:
Patch Localization via Inversion on a Trained (Frozen) Neural Field

The goal of this task is to determine where a given image patch (a small region) is located within the coordinate space of a trained neural field. Unlike Task 1, which inverts a single (x, y) coordinate to find a target color, here we invert an entire **patch**: we optimize its **center** (cx, cy) and **scale** (sx, sy) so that the neural field, sampled over the corresponding grid of coordinates, reproduces the target patch as accurately as possible.

Step-by-step procedure:

1. Train (or load) a neural field model (e.g., SIREN) on a full image and freeze its parameters.
   - Easy: use `map-saturation.png` and its corresponding patch `map-saturation-patch.png`.
   - Medium: use `lego-bricks.png` and `lego-bricks-patch.png`.
   - Difficult: use `cats.jpeg` and `cats-patch.jpeg`.
   (You may want to adjust the neural field's parameters accordingly.)

2. Load a **target patch**—a small crop from an image (e.g., from a .jpeg file such as `lego-bricks-patch.jpeg`). This patch is the pattern you want to localize within the neural field’s coordinate space.

3. Parameterize the patch location using a **center** (cx, cy) and **scale** (sx, sy), each in the normalized range [-1, 1]. Implement a differentiable function that, given (cx, cy, sx, sy), produces the grid of (x, y) coordinates corresponding to the patch (for example, a local grid in [-1, 1] that is scaled and shifted by the center and scale).

4. Initialize learnable parameters for the patch (such as center and log-scale), and set up an optimizer that updates only these parameters while the neural field remains frozen.

5. In the optimization loop:
   - Consider different loss (e.g., MSE, or a perceptual loss such as VGG, or another suitable loss), compare them.

6. Visualize the results:
   - Plot the loss over iterations.
   - Show the target patch alongside the recovered patch (the patch sampled from the field at the optimized location).
   - Overlay the recovered patch location (for example, as a bounding box) on the full image to show where the patch was found.

7. You should consider trying different random initializations for (cx, cy, sx, sy), as the optimization problem is highly non-convex in some cases. 
Discuss whether the optimization converges to the same or to different locations.

In [ ]:
import torch.nn.functional as F

img_path = "lego-bricks.jpeg"
resolution = 512 
img_np = load_image(img_path, resize=(resolution, resolution))

coords, colors = prepare_dataset(img_np)

hidden_dim_siren = 256
num_layers_siren = 5
omega_0 = 30.0
# Train and plot the training dynamics of the SIREN neural field model.
model = SIREN(in_dim=2, hidden_dim=hidden_dim_siren, out_dim=3, num_layers=num_layers_siren, omega_0=omega_0).to(device)
num_iters = 2000
lr = 1e-4
trained_model, history = train_model(model, coords, colors, num_iters=num_iters, lr=lr, loss_type="mse", log_every=200)


def get_patch_grid(center, scale, patch_res=32, device='cuda'):
    """
    Génère une grille de coordonnées (N, patch_res*patch_res, 2)
    center: (N, 2) -> [cx, cy]
    scale:  (N, 2) -> [sx, sy]
    """
    # Grille locale de -1 à 1
    t = torch.linspace(-1, 1, patch_res, device=device)
    grid_y, grid_x = torch.meshgrid(t, t, indexing='ij')
    local_grid = torch.stack([grid_x, grid_y], dim=-1).reshape(-1, 2) # (P*P, 2)
    
    # Transformation : Global = Center + Scale * Local
    # On utilise scale/2 car la largeur totale du patch est 2*scale dans l'espace [-1,1]
    global_grid = center.unsqueeze(1) + (scale.unsqueeze(1) / 2.0) * local_grid.unsqueeze(0)
    return global_grid

# --- Initialisation ---
target_patch_img = load_image("lego-bricks-patch.jpeg", resize=(32, 32))
target_patch_tensor = torch.from_numpy(target_patch_img).to(device).reshape(-1, 3)

# Paramètres apprenables
# On initialise souvent l'échelle à une petite valeur (ex: 0.2)
center = torch.zeros((1, 2), device=device, requires_grad=True)
log_scale = torch.log(torch.tensor([0.2, 0.2], device=device)).unsqueeze(0).requires_grad_(True)

optimizer = torch.optim.Adam([center, log_scale], lr=5e-3)

# --- Boucle d'optimisation ---
trained_model.eval()
for param in trained_model.parameters():
    param.requires_grad = False

for i in range(1500):
    optimizer.zero_grad()
    
    curr_scale = torch.exp(log_scale) # On utilise log_scale pour garantir une échelle > 0
    grid = get_patch_grid(center, curr_scale, patch_res=32, device=device)
    
    # Échantillonnage du champ de neurones
    pred_patch = trained_model(grid.reshape(-1, 2))
    
    loss = F.mse_loss(pred_patch, target_patch_tensor)
    loss.backward()
    optimizer.step()
    
    # Projection pour rester dans l'image [-1, 1]
    with torch.no_grad():
        center.clamp_(-1, 1)

    if i % 300 == 0:
        print(f"Iter {i}, Loss: {loss.item():.6f}, Scale: {curr_scale.detach().cpu().numpy()}")

# Affichage des résultats
final_scale = torch.exp(log_scale).detach().cpu().numpy()[0]
final_center = center.detach().cpu().numpy()[0]
print(f"Final Center: {final_center}, Final Scale: {final_scale}")
grid = get_patch_grid(center, torch.exp(log_scale), patch_res=32, device=device)
with torch.no_grad():
    pred_patch = trained_model(grid.reshape(-1, 2)).cpu().numpy().reshape(32, 32, 3)    
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.title("Target Patch")
plt.imshow(target_patch_img)
plt.axis('off')
plt.subplot(1,2,2)  
plt.title("Recovered Patch")
plt.imshow(pred_patch)
plt.axis('off')
plt.show()

In [ ]:
import matplotlib.patches as patches

def visualize_inversion_results(full_img, target_patch, model, center, log_scale, resolution=128):
    model.eval()
    with torch.no_grad():
        curr_scale = torch.exp(log_scale)
        # 1. Récupérer le patch reconstruit à la position optimisée
        grid = get_patch_grid(center, curr_scale, patch_res=target_patch.shape[0], device=device)
        recon_patch = model(grid.reshape(-1, 2)).reshape(target_patch.shape).cpu().numpy()
        
        # 2. Convertir les coordonnées normalisées [-1, 1] en coordonnées pixels [0, res]
        cx, cy = center[0].cpu().numpy()
        sx, sy = curr_scale[0].cpu().numpy()
        
        # Le coin supérieur gauche (x, y) pour le rectangle matplotlib
        # x_pixel = (coord_norm + 1) * (res / 2)
        x_min = (cx - sx / 2 + 1) * (resolution / 2)
        y_min = (cy - sy / 2 + 1) * (resolution / 2)
        width = sx * (resolution / 2)
        height = sy * (resolution / 2)

    # Affichage
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    # Subplot 1: Image complète + Bounding Box
    ax[0].imshow(full_img)
    rect = patches.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor='r', facecolor='none', label='Localized Patch')
    ax[0].add_patch(rect)
    ax[0].set_title("Full Image & Localization")
    ax[0].axis('off')

    # Subplot 2: Patch Cible (Ground Truth)
    ax[1].imshow(target_patch)
    ax[1].set_title("Target Patch (GT)")
    ax[1].axis('off')

    # Subplot 3: Patch extrait du Neural Field
    ax[2].imshow(recon_patch)
    ax[2].set_title("Recovered Patch (Inversion)")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

visualize_inversion_results(img_np, target_patch_img, trained_model, center, log_scale, resolution)

Le paysage de perte pour la localisation de patch est semé de minima locaux pour deux raisons :
- Comme SIREN utilise des sinus, si l'initialisation est trop loin du patch réel, l'optimiseur peut se stabiliser sur une zone de couleur moyenne similaire.
- Dans des images répétitives (comme lego-bricks), plusieurs positions peuvent minimiser la MSE localement.

## Bonus:
Speed up: 
- Do we really need to compare all pixels?
- How to do coarse-to-fine? (can we move the patch?)
- How to decouple chromatic and the shape?

Try some and have fun ;)

On peut travailler avec des processus stochatsiques et ne prendre qu'une partie. 

## Task3 — GAN inversion + CLIP manipulation

This section gives a **minimal** example of:
1. **GAN inversion**: find a latent code $z$ such that the generator $G(z)$ reconstructs a target image (we optimize $z$ to minimize $\mathcal{L}(G(z), x)$).
2. **CLIP manipulation**: move $z$ so that $G(z)$ better matches a text prompt, by maximizing CLIP similarity between the generated image and the text.

We use **Hugging Face** models:
- **StyleGAN2** (face generator, 128×128): `hajar001/stylegan2-ffhq-128`
- **CLIP** (image–text similarity): `openai/clip-vit-base-patch32` via `transformers`

Dependencies: `pip install transformers huggingface_hub` (plus `torch`, `PIL`, `matplotlib` from the rest of the lab).

Objective: recover a StyleGAN2 latent that reproduces a target face, then steer the latent toward a text concept using CLIP.

What to do:
1. Load the pretrained StyleGAN2 generator and CLIP model; prepare a 128×128 target face (sampled or external).
2. Optimize a latent vector so the generated image matches the target in pixel space.
3. Starting from the inverted latent, run a second optimization that maximizes CLIP similarity to a chosen prompt while regularizing drift from the inversion.
4. Visualize target, inversion, and edited outputs side by side and describe how the prompt changes appearance.

Required packages: `torch`, `transformers`, `huggingface_hub`, `PIL`, `matplotlib`.


In [ ]:
# Task3: GAN inversion + CLIP manipulation — load models from Hugging Face

import os
import sys
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

from huggingface_hub import hf_hub_download

style_gan_file = hf_hub_download(repo_id="hajar001/stylegan2-ffhq-128", filename="style_gan.py")
sys.path.insert(0, os.path.dirname(style_gan_file))
from style_gan import StyleGAN

gan = StyleGAN.from_pretrained("hajar001/stylegan2-ffhq-128").to(device).eval()
z_dim = 512

In [ ]:
# Generate a target face from random z (we will invert this and then manipulate with CLIP)
z0 = torch.randn(1, z_dim, device=device)

with torch.no_grad():
    img_target, _ = gan(z0)
    # GAN output is in [-1, 1]; convert to [0, 1] for display and for inversion target
    img_target = (img_target + 1) / 2
    img_target = torch.clamp(img_target, 0, 1)



# or take directly from the repo
image_dir = "cats.jpeg"
img_target = np.asarray(Image.open(image_dir).convert("RGB"), dtype=np.float32) / 255.0
# center crop and resize to 128x128
img_pil = Image.fromarray((np.clip(img_target, 0, 1) * 255).astype(np.uint8))
img_target = np.asarray(img_pil.resize((128, 128)), dtype=np.float32) / 255.0
# (H, W, C) -> (1, C, H, W) for PyTorch: transpose to (C, H, W) then add batch
img_target = torch.from_numpy(img_target.transpose(2, 0, 1)[np.newaxis, ...].copy()).float().to(device)

plt.figure(figsize=(4, 4))
plt.imshow(img_target[0].permute(1, 2, 0).cpu().numpy())
plt.title("Target face G(z0)")
plt.axis("off")
plt.tight_layout()
plt.show()
print("Target image shape:", img_target.shape)  # (1, 3, 128, 128)


In [ ]:
# 3) GAN inversion: optimize latent z to match target
# Your code here
z_optimized = torch.randn(1, z_dim, device=device, requires_grad=True)

# 2. Définition de l'optimiseur (Adam est le standard ici)
optimizer = torch.optim.Adam([z_optimized], lr=0.01)

num_iters = 500
history_loss = []

print("Starting GAN inversion...")

for i in range(1, num_iters + 1):
    optimizer.zero_grad()
    
    # Génération de l'image à partir du z courant
    img_gen, _ = gan(z_optimized)
    
    # Normalisation : Sortie GAN [-1, 1] -> [0, 1] pour correspondre à img_target
    img_gen = (img_gen + 1) / 2
    img_gen = torch.clamp(img_gen, 0, 1)
    
    # Calcul de la perte MSE (Pixel-wise loss)
    loss = F.mse_loss(img_gen, img_target)
    
    # Backpropagation
    loss.backward()
    optimizer.step()
    
    history_loss.append(loss.item())
    
    if i % 100 == 0:
        print(f"Iteration {i}/{num_iters}, Loss: {loss.item():.6f}")

# Visualisation du résultat de l'inversion
with torch.no_grad():
    final_img, _ = gan(z_optimized)
    final_img = torch.clamp((final_img + 1) / 2, 0, 1)

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_target[0].permute(1, 2, 0).cpu().numpy())
plt.title("Target")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(final_img[0].permute(1, 2, 0).cpu().numpy())
plt.title("Reconstructed (Inverted)")
plt.axis("off")
plt.show()

We are in a OoD scenario.

In [ ]:
# 4) CLIP manipulation: move z to match a text prompt
# Your code here
prompt = "a person with blue hair and sunglasses"
text_inputs = clip_processor(text=[prompt], return_tensors="pt", padding=True).to(device)

# 2. Extract text features (reference)
with torch.no_grad():
    outputs = clip_model.get_text_features(**text_inputs)
    
    # If outputs is still a BaseModelOutput object, use outputs.pooler_output
    if hasattr(outputs, "pooler_output"):
        text_features = outputs.pooler_output
    else:
        text_features = outputs # It is already a tensor in some versions
        
    # Now the norm operation will work
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

# 3. Use the previously inverted z as a starting point
z_manipulated = z_optimized.detach().clone().requires_grad_(True)
optimizer = torch.optim.Adam([z_manipulated], lr=0.02)

# Parameters for CLIP
clip_size = 224 # CLIP ViT-B/32 resolution
num_iters = 200

for i in range(1, num_iters + 1):
    optimizer.zero_grad()
    
    # Generate image
    img_gen, _ = gan(z_manipulated)
    img_gen = (img_gen + 1) / 2 # [-1, 1] -> [0, 1]
    
    # Resize to CLIP expected input size
    img_resized = F.interpolate(img_gen, size=(clip_size, clip_size), mode='bilinear', align_corners=False)
    
    # Normalize with ImageNet stats (required for CLIP)
    # Mean: [0.481, 0.457, 0.408], Std: [0.268, 0.261, 0.275]
    img_norm = (img_resized - torch.tensor([0.481, 0.457, 0.408], device=device).view(1, 3, 1, 1)) / \
                torch.tensor([0.268, 0.261, 0.275], device=device).view(1, 3, 1, 1)
    
    # Get image features
    image_features = clip_model.get_image_features(img_norm)
    if hasattr(image_features, "pooler_output"):
        image_features = image_features.pooler_output
    else:
        image_features = image_features
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    
    # Calculate CLIP loss (negative cosine similarity)
    loss = -torch.cosine_similarity(image_features, text_features).mean()
    
    # Optional: Add L2 penalty to stay close to original face manifold
    reg_loss = F.mse_loss(z_manipulated, z_optimized.detach()) * 0.5
    total_loss = loss + reg_loss
    
    total_loss.backward()
    optimizer.step()
    
    if i % 50 == 0:
        print(f"Iter {i}/{num_iters}, CLIP Loss: {loss.item():.4f}")

# Final visualization
with torch.no_grad():
    manipulated_img, _ = gan(z_manipulated)
    manipulated_img = torch.clamp((manipulated_img + 1) / 2, 0, 1)

plt.figure(figsize=(4, 4))
plt.imshow(manipulated_img[0].permute(1, 2, 0).cpu().numpy())
plt.title(f"Result: '{prompt}'")
plt.axis("off")
plt.show()

## Discussion:
Are you satisfied with the results? If not, why? What potential methods could improve the results? (Just answer here; implementation is not required.)

Answer: I am not statisfied